In [30]:
import pandas as pd

In [32]:
df = pd.read_csv("../data/raw/gender.csv")

In [33]:
df_option_1 = df
df_option_2 = df
df_option_3 = df
df

,auhtor_ID,post,female
0,t2_rnjzutp,Good on you for being responsible! I know self...,1
1,t2_rnjzutp,"must go to the grocery store with their child,...",1
2,t2_rnjzutp,"things on her videos, and YouTube took the vid...",1
3,t2_rnjzutp,their app. There's also a program called SYNC ...,1
4,t2_rnjzutp,"side. If the cops don't take your side, you'll...",1
...,...,...,...
44630,t2_6mpla2l0,if smegma kept her kids away just out of spite...,1
44631,t2_6mpla2l0,PhDs to change the time on my microwave. I did...,1
44632,t2_6mpla2l0,HiLIARy could even think of doing! I think Car...,1
44633,t2_6mpla2l0,of the hand is a breeze. It swells after thoug...,1


In [34]:
import re
import unicodedata
import emoji

# Define allowed characters:
# - ASCII letters, digits, punctuation, whitespace
# - Emojis (checked via emoji library)
# - Zero Width Joiner (u200d)
# - Variation Selector-16 (uFE0F) if you want to keep color variants of emojis

def is_allowed_char(ch):
    # Check if ASCII (common)
    if ch.isascii():
        return True
    # Keep zero-width joiner
    if ch == '\u200d':
        return True
    # Keep variation selector if needed
    if ch == '\uFE0F':
        return True
    # Check if char is an emoji
    # The 'emoji' library: emoji.is_emoji returns True if it's a single emoji
    if emoji.is_emoji(ch):
        return True
    
    # Otherwise, exclude
    return False

def clean_text(text):
    # Iterate over each character and keep only allowed ones
    return ''.join(ch for ch in text if is_allowed_char(ch))

# Example usage:
post = "Hi👩\u200d👩\u200d👧\u200d👦! ©®™¥•√π÷×§∆ This should stay: 😂👩\u200d💻 and remove ﷽ Arabic: مرحبا"
cleaned = clean_text(post)
print(cleaned)

Hi👩‍👩‍👧‍👦! ©®™ This should stay: 😂👩‍💻 and remove  Arabic: 


In [35]:
import pandas as pd
import emoji

# Define which characters are allowed:
# - ASCII letters, digits, punctuation, whitespace
# - Emojis
# - Zero Width Joiner (\u200d) and Variation Selector-16 (\uFE0F)

def is_allowed_char(ch):
    # Check if it's an ASCII character (includes basic letters, digits, punctuation)
    if ch.isascii():
        return True
    # Keep zero-width joiner used for complex emoji
    if ch == '\u200d':
        return True
    # Keep variation selector, often used with emojis
    if ch == '\uFE0F':
        return True
    # Keep if it's an emoji
    if emoji.is_emoji(ch):
        return True
    # Otherwise, exclude it
    return False

def clean_text(text):
    # Filter out characters not allowed
    return ''.join(ch for ch in text if is_allowed_char(ch))

# Apply cleaning to each post
df_option_1['clean_post'] = df['post'].apply(clean_text)

# Determine how many posts changed
df_option_1['changed'] = (df_option_1['clean_post'] != df_option_1['post'])
changed_count = df_option_1['changed'].sum()
total_count = len(df)
changed_percentage = (changed_count / total_count) * 100

print(f"Number of posts changed: {changed_count}")
print(f"Percentage of posts changed: {changed_percentage:.2f}%")
df_option_1.to_csv('df_option_1.csv', index=False)

Number of posts changed: 26800
Percentage of posts changed: 60.04%


In [36]:
import pandas as pd
import unicodedata
import emoji

def is_allowed_char(ch):
    # Zero Width Joiner and Variation Selector
    if ch in ['\u200d', '\uFE0F']:
        return True
    
    # Check if it's an emoji
    if emoji.is_emoji(ch):
        return True
    
    # Check if it's an ASCII character
    if ch.isascii():
        # Get the Unicode category of the character
        cat = unicodedata.category(ch)
        # Allow letters, digits, punctuation, and spaces from ASCII set
        # Categories of interest:
        # L* = Letters, N* = Numbers, P* = Punctuation, Zs = Space Separator
        # We'll also include M (Mark) if needed, but usually ASCII won't have combining marks.
        if cat.startswith(('L', 'N', 'P')) or cat == 'Zs':
            return True
    
    return False

def clean_text(text):
    return ''.join(ch for ch in text if is_allowed_char(ch))

# Clean posts
df_option_2['clean_post'] = df['post'].apply(clean_text)

# Determine which posts changed
df_option_2['changed'] = (df_option_2['clean_post'] != df_option_2['post'])
changed_count = df_option_2['changed'].sum()
total_count = len(df)
changed_percentage = (changed_count / total_count) * 100

print(f"Number of posts changed: {changed_count}")
print(f"Percentage of posts changed: {changed_percentage:.2f}%")
df_option_2.to_csv('df_option_2.csv', index=False)

Number of posts changed: 36080
Percentage of posts changed: 80.83%


In [40]:
import pandas as pd
import re

# Define allowed Unicode ranges:
# Basic Latin (U+0000–U+007F)
# Latin-1 Supplement (U+0080–U+00FF) - If you need just ASCII, you can skip this.
# Common emoji ranges:
#   Emoticons: U+1F600–U+1F64F
#   Misc. Symbols & Pictographs: U+1F300–U+1F5FF
#   Supplemental Symbols & Pictographs: U+1F900–U+1FAFF
#   Flags: U+1F1E0–U+1F1FF
# Also include Zero Width Joiner (U+200D) and Variation Selector-16 (U+FE0F)

allowed_pattern = re.compile(
    r"[\u0000-\u00FF]"               # Basic Latin + Latin-1
    r"|[\U0001F300-\U0001F5FF]"      # Misc. Symbols & Pictographs
    r"|[\U0001F600-\U0001F64F]"      # Emoticons
    r"|[\U0001F900-\U0001FAFF]"      # Supplemental Symbols & Pictographs
    r"|[\U0001F1E0-\U0001F1FF]"      # Flags
    r"|[\u200d\uFE0F]"               # Zero Width Joiner + Variation Selector-16
)

def clean_text(text):
    # Keep only allowed characters
    return ''.join(ch for ch in text if allowed_pattern.match(ch))

df_option_3['clean_post'] = df['post'].apply(clean_text)

# Find how many changed
df_option_3['changed'] = (df_option_3['clean_post'] != df_option_3['post'])
changed_count = df_option_3['changed'].sum()
total_count = len(df)
changed_percentage = (changed_count / total_count) * 100

print(f"Number of posts changed: {changed_count}")
print(f"Percentage of posts changed: {changed_percentage:.2f}%")
df_option_3.to_csv('df_option_3.csv', index=False)

Number of posts changed: 26090
Percentage of posts changed: 58.45%


In [44]:
import pandas as pd

# Load the saved CSV files back into DataFrames
df_option_1 = pd.read_csv('df_option_1.csv')
df_option_2 = pd.read_csv('df_option_2.csv')
df_option_3 = pd.read_csv('df_option_3.csv')

In [45]:
df_option_1

,auhtor_ID,post,female,clean_post,changed
0,t2_rnjzutp,Good on you for being responsible! I know self...,1,Good on you for being responsible! I know self...,False
1,t2_rnjzutp,"must go to the grocery store with their child,...",1,"must go to the grocery store with their child,...",False
2,t2_rnjzutp,"things on her videos, and YouTube took the vid...",1,"things on her videos, and YouTube took the vid...",False
3,t2_rnjzutp,their app. There's also a program called SYNC ...,1,their app. There's also a program called SYNC ...,True
4,t2_rnjzutp,"side. If the cops don't take your side, you'll...",1,"side. If the cops don't take your side, you'll...",False
...,...,...,...,...,...
44630,t2_6mpla2l0,if smegma kept her kids away just out of spite...,1,if smegma kept her kids away just out of spite...,True
44631,t2_6mpla2l0,PhDs to change the time on my microwave. I did...,1,PhDs to change the time on my microwave. I did...,True
44632,t2_6mpla2l0,HiLIARy could even think of doing! I think Car...,1,HiLIARy could even think of doing! I think Car...,True
44633,t2_6mpla2l0,of the hand is a breeze. It swells after thoug...,1,of the hand is a breeze. It swells after thoug...,True


In [46]:
df_option_2

,auhtor_ID,post,female,clean_post,changed
0,t2_rnjzutp,Good on you for being responsible! I know self...,1,Good on you for being responsible! I know self...,False
1,t2_rnjzutp,"must go to the grocery store with their child,...",1,"must go to the grocery store with their child,...",False
2,t2_rnjzutp,"things on her videos, and YouTube took the vid...",1,"things on her videos, and YouTube took the vid...",True
3,t2_rnjzutp,their app. There's also a program called SYNC ...,1,their app. There's also a program called SYNC ...,True
4,t2_rnjzutp,"side. If the cops don't take your side, you'll...",1,"side. If the cops don't take your side, you'll...",False
...,...,...,...,...,...
44630,t2_6mpla2l0,if smegma kept her kids away just out of spite...,1,if smegma kept her kids away just out of spite...,True
44631,t2_6mpla2l0,PhDs to change the time on my microwave. I did...,1,PhDs to change the time on my microwave. I did...,True
44632,t2_6mpla2l0,HiLIARy could even think of doing! I think Car...,1,HiLIARy could even think of doing! I think Car...,True
44633,t2_6mpla2l0,of the hand is a breeze. It swells after thoug...,1,of the hand is a breeze. It swells after thoug...,True


In [47]:
df_option_3

,auhtor_ID,post,female,clean_post,changed
0,t2_rnjzutp,Good on you for being responsible! I know self...,1,Good on you for being responsible! I know self...,False
1,t2_rnjzutp,"must go to the grocery store with their child,...",1,"must go to the grocery store with their child,...",False
2,t2_rnjzutp,"things on her videos, and YouTube took the vid...",1,"things on her videos, and YouTube took the vid...",False
3,t2_rnjzutp,their app. There's also a program called SYNC ...,1,their app. There's also a program called SYNC ...,False
4,t2_rnjzutp,"side. If the cops don't take your side, you'll...",1,"side. If the cops don't take your side, you'll...",False
...,...,...,...,...,...
44630,t2_6mpla2l0,if smegma kept her kids away just out of spite...,1,if smegma kept her kids away just out of spite...,True
44631,t2_6mpla2l0,PhDs to change the time on my microwave. I did...,1,PhDs to change the time on my microwave. I did...,True
44632,t2_6mpla2l0,HiLIARy could even think of doing! I think Car...,1,HiLIARy could even think of doing! I think Car...,True
44633,t2_6mpla2l0,of the hand is a breeze. It swells after thoug...,1,of the hand is a breeze. It swells after thoug...,True


In [52]:
# Ensure both DataFrames are aligned row-by-row for comparison
common_true_rows = (df_option_1['changed'] & df_option_2['changed']).sum()

# Total "changed" rows in each option
total_true_option1 = df_option_1['changed'].sum()
total_true_option2 = df_option_2['changed'].sum()

# Calculate percentage of common "changed" rows relative to each option
percent_common_from_option1 = (common_true_rows / total_true_option1 * 100) if total_true_option1 > 0 else 0
percent_common_from_option2 = (common_true_rows / total_true_option2 * 100) if total_true_option2 > 0 else 0

# Create a results DataFrame
result_df = pd.DataFrame({
    'common_changed_rows': [common_true_rows],
    '%_common_from_option1': [percent_common_from_option1],
    '%_common_from_option2': [percent_common_from_option2]
})

# Display the results
result_df

,common_changed_rows,%_common_from_option1,%_common_from_option2
0,26800,100.0,74.279379


In [54]:
# Ensure both DataFrames are aligned row-by-row for comparison
common_true_rows_2_3 = (df_option_2['changed'] & df_option_3['changed']).sum()

# Total "changed" rows in each option
total_true_option2 = df_option_2['changed'].sum()
total_true_option3 = df_option_3['changed'].sum()

# Calculate percentage of common "changed" rows relative to each option
percent_common_from_option2 = (common_true_rows_2_3 / total_true_option2 * 100) if total_true_option2 > 0 else 0
percent_common_from_option3 = (common_true_rows_2_3 / total_true_option3 * 100) if total_true_option3 > 0 else 0

# Create a results DataFrame
result_df_2_3 = pd.DataFrame({
    'common_changed_rows': [common_true_rows_2_3],
    '%_common_from_option2': [percent_common_from_option2],
    '%_common_from_option3': [percent_common_from_option3]
})

# Display the results
print(result_df_2_3)

   common_changed_rows  %_common_from_option2  %_common_from_option3
0                25042              69.406874              95.983135


In [56]:
# Ensure both DataFrames are aligned row-by-row for comparison
common_true_rows_1_3 = (df_option_1['changed'] & df_option_3['changed']).sum()

# Total "changed" rows in each option
total_true_option1 = df_option_1['changed'].sum()
total_true_option3 = df_option_3['changed'].sum()

# Calculate percentage of common "changed" rows relative to each option
percent_common_from_option1 = (common_true_rows_1_3 / total_true_option1 * 100) if total_true_option1 > 0 else 0
percent_common_from_option3 = (common_true_rows_1_3 / total_true_option3 * 100) if total_true_option3 > 0 else 0

# Create a results DataFrame
result_df_1_3 = pd.DataFrame({
    'common_changed_rows': [common_true_rows_1_3],
    '%_common_from_option1': [percent_common_from_option1],
    '%_common_from_option3': [percent_common_from_option3]
})

# Display the results
print(result_df_1_3)

   common_changed_rows  %_common_from_option1  %_common_from_option3
0                23850              88.992537              91.414335


In [58]:
# Ensure all DataFrames are aligned row-by-row for comparison
common_true_rows_1_2_3 = (df_option_1['changed'] & df_option_2['changed'] & df_option_3['changed']).sum()

# Total "changed" rows in each option
total_true_option1 = df_option_1['changed'].sum()
total_true_option2 = df_option_2['changed'].sum()
total_true_option3 = df_option_3['changed'].sum()

# Calculate percentage of common "changed" rows relative to each option
percent_common_from_option1 = (common_true_rows_1_2_3 / total_true_option1 * 100) if total_true_option1 > 0 else 0
percent_common_from_option2 = (common_true_rows_1_2_3 / total_true_option2 * 100) if total_true_option2 > 0 else 0
percent_common_from_option3 = (common_true_rows_1_2_3 / total_true_option3 * 100) if total_true_option3 > 0 else 0

# Create a results DataFrame
result_df_1_2_3 = pd.DataFrame({
    'common_changed_rows': [common_true_rows_1_2_3],
    '%_common_from_option1': [percent_common_from_option1],
    '%_common_from_option2': [percent_common_from_option2],
    '%_common_from_option3': [percent_common_from_option3]
})

# Display the results
print(result_df_1_2_3)

   common_changed_rows  %_common_from_option1  %_common_from_option2  \
0                23850              88.992537              66.103104   

   %_common_from_option3  
0              91.414335  


In [62]:
# Select 3 examples of posts where "changed" is True in df_option_1
example_indices = df_option_1[df_option_1['changed']].head(3).index

# Create a DataFrame to compare the original post and its cleaned versions
comparison_df = pd.DataFrame({
    'original_post': df_option_1.loc[example_indices, 'post'],  # Original post
    'cleaned_by_option1': df_option_1.loc[example_indices, 'clean_post'],  # Cleaned by option1
    'cleaned_by_option2': df_option_2.loc[example_indices, 'clean_post'],  # Cleaned by option2
    'cleaned_by_option3': df_option_3.loc[example_indices, 'clean_post'],  # Cleaned by option3
})

comparison_df

,original_post,cleaned_by_option1,cleaned_by_option2,cleaned_by_option3
3,their app. There's also a program called SYNC ...,their app. There's also a program called SYNC ...,their app. There's also a program called SYNC ...,their app. There's also a program called SYNC ...
5,but the mind fog and exhaustion afterwards fit...,but the mind fog and exhaustion afterwards fit...,but the mind fog and exhaustion afterwards fit...,but the mind fog and exhaustion afterwards fit...
9,people for around $30 or $40 USD if I remember...,people for around $30 or $40 USD if I remember...,people for around 30 or 40 USD if I remember c...,people for around $30 or $40 USD if I remember...


In [92]:
comparison_df['original_post'].iloc[0]

'their app. There\'s also a program called SYNC that runs over the summer and lends audiobooks to teenagers to encourage literacy. It\'s entirely 100% free. I was a member back when I was still in high school. Here\'s the link if you\'d like to get her set up on it for the 2023 season: url Consider games as well. It may not seem like people learn much from playing games, but it teaches patience, empathy, and how to be a good sport. I personally like the app Plato because it\'s free and has a lot of off-brand versions of classic games. This would also be a way for you to connect with her while she\'s away. These are just suggestions though, and they aren\'t a good fit for every person. I hope it works out well for all of you. Leave uta no prince sama out of this /joking All Lady D content reminds me of that one song they used in the movie Igor (2008). The Bigger the Figure by Louis Prima. I (21F) just took a driving class last month because I\'m finally getting my license. I\'m getting 

In [96]:
comparison_df['cleaned_by_option1'].iloc[0]

'their app. There\'s also a program called SYNC that runs over the summer and lends audiobooks to teenagers to encourage literacy. It\'s entirely 100% free. I was a member back when I was still in high school. Here\'s the link if you\'d like to get her set up on it for the 2023 season: url Consider games as well. It may not seem like people learn much from playing games, but it teaches patience, empathy, and how to be a good sport. I personally like the app Plato because it\'s free and has a lot of off-brand versions of classic games. This would also be a way for you to connect with her while she\'s away. These are just suggestions though, and they aren\'t a good fit for every person. I hope it works out well for all of you. Leave uta no prince sama out of this /joking All Lady D content reminds me of that one song they used in the movie Igor (2008). The Bigger the Figure by Louis Prima. I (21F) just took a driving class last month because I\'m finally getting my license. I\'m getting 

In [102]:
import re

# Function to remove ASCII art-like patterns
def remove_ascii_art(text):
    # Split text into lines and filter lines that don't look like ASCII art
    lines = text.splitlines()
    filtered_lines = [
        line for line in lines
        # Match lines with excessive symbols or non-alphanumeric content
        if not re.match(r"^[\s./\\|_`'\":;~*!@#$%^&*(){}[\]<>-]{3,}$", line) and len(line.strip()) > 0
    ]
    # Join filtered lines back into text
    return "\n".join(filtered_lines)

# Apply the ASCII art removal function to the cleaned posts in df_option_2
df_final = df_option_2.copy()
df_final['final_clean_post'] = df_final['clean_post'].apply(remove_ascii_art)

# Determine which posts were corrected in the final cleaning
df_final['corrected'] = (df_final['final_clean_post'] != df_final['clean_post'])
corrected_count = df_final['corrected'].sum()
total_count = len(df_option_2)
corrected_percentage = (corrected_count / total_count) * 100

# Print statistics
print(f"Number of posts corrected: {corrected_count}")
print(f"Percentage of posts corrected: {corrected_percentage:.2f}%")

# Save the final DataFrame
df_final.to_csv('df_final.csv', index=False)

Number of posts corrected: 0
Percentage of posts corrected: 0.00%


In [99]:
df_final

,auhtor_ID,post,female,clean_post,changed,final_clean_post,corrected
0,t2_rnjzutp,Good on you for being responsible! I know self...,1,Good on you for being responsible! I know self...,False,Good on you for being responsible! I know self...,False
1,t2_rnjzutp,"must go to the grocery store with their child,...",1,"must go to the grocery store with their child,...",False,"must go to the grocery store with their child,...",False
2,t2_rnjzutp,"things on her videos, and YouTube took the vid...",1,"things on her videos, and YouTube took the vid...",True,"things on her videos, and YouTube took the vid...",False
3,t2_rnjzutp,their app. There's also a program called SYNC ...,1,their app. There's also a program called SYNC ...,True,their app. There's also a program called SYNC ...,False
4,t2_rnjzutp,"side. If the cops don't take your side, you'll...",1,"side. If the cops don't take your side, you'll...",False,"side. If the cops don't take your side, you'll...",False
...,...,...,...,...,...,...,...
44630,t2_6mpla2l0,if smegma kept her kids away just out of spite...,1,if smegma kept her kids away just out of spite...,True,if smegma kept her kids away just out of spite...,False
44631,t2_6mpla2l0,PhDs to change the time on my microwave. I did...,1,PhDs to change the time on my microwave. I did...,True,PhDs to change the time on my microwave. I did...,False
44632,t2_6mpla2l0,HiLIARy could even think of doing! I think Car...,1,HiLIARy could even think of doing! I think Car...,True,HiLIARy could even think of doing! I think Car...,False
44633,t2_6mpla2l0,of the hand is a breeze. It swells after thoug...,1,of the hand is a breeze. It swells after thoug...,True,of the hand is a breeze. It swells after thoug...,False


In [104]:
# Compare the 'clean_post' column in df_option_2 with the 'final_clean_post' column in df_final
are_equal = df_option_2['clean_post'].equals(df_final['final_clean_post'])

# Print the result of the comparison
if are_equal:
    print("The 'clean_post' column in df_option_2 is identical to the 'final_clean_post' column in df_final.")
else:
    print("The 'clean_post' column in df_option_2 is NOT identical to the 'final_clean_post' column in df_final.")

# Check how many rows are different
differences = (df_option_2['clean_post'] != df_final['final_clean_post']).sum()
print(f"Number of rows with differences: {differences}")

# Optional: Display rows with differences
diff_rows = df_option_2.loc[df_option_2['clean_post'] != df_final['final_clean_post']]
print("Rows with differences:")
print(diff_rows)

The 'clean_post' column in df_option_2 is identical to the 'final_clean_post' column in df_final.
Number of rows with differences: 0
Rows with differences:
Empty DataFrame
Columns: [auhtor_ID, post, female, clean_post, changed]
Index: []
